# GéoMarketing IDF : Mobilités professionnelles et scolaires à la commune en Île-de-France

## Objectifs
- Mesurer les liens entre communes de résidence et communes de travail ou d’études.
- Distinguer les effectifs pondérés de résidents, les internes, les entrants et les sortants.
- Conserver les échanges avec les communes hors Île-de-France et les destinations étrangères.
- Étudier les modes de transport domicile-travail et les âges des personnes scolarisées.
- Comparer Cergy et Pontoise, puis analyser leur périmètre commun sans doubles comptages.
- Conserver tous les IRIS de J13c et leur associer un contexte communal explicitement identifié.
- Exporter des CSV, GeoPackages, cartes Plotly et documents légers pour GitHub.
## Périmètre
RP 2022, géographie communale au 1er janvier 2024. Les deux fichiers détail sont
nationaux ; une observation est retenue dès que sa résidence OU sa destination
concerne une commune IDF. Le focus initial porte sur les communes de Cergy (95127)
et Pontoise (95500), et non sur toute l’agglomération.
## Interprétation
Les résultats sont des effectifs de personnes estimés avec IPONDI, et non des
nombres de trajets quotidiens ou des passages en gare. Une personne peut figurer
dans les deux fichiers : travail et études ne sont jamais additionnés pour obtenir
une clientèle unique. Le lieu de travail déclaré ne donne pas la présence chaque
midi ni le télétravail. L’âge 18–24 ans des personnes scolarisées ne prouve pas une
inscription dans l’enseignement supérieur ni dans un campus précis.
Les flux sont communaux. Ils sont joints aux IRIS comme CONTEXTE COMMUNAL ; aucun
flux n’est réparti entre IRIS au prorata des habitants. Les tracés des cartes sont
schématiques entre points représentatifs des communes, pas des itinéraires.

In [2]:
#Import et préparation de l'environnement

import os
for variable in ['OMP_NUM_THREADS','NUMEXPR_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[variable]='1'
from pathlib import Path
from zipfile import ZipFile
from contextlib import contextmanager
from collections import defaultdict
from datetime import datetime, timezone
import csv, hashlib, html, json, math, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly
import plotly.graph_objects as go
from IPython.display import display
pd.set_option('display.max_columns',70)
pd.set_option('display.max_rows',30)
print('pandas',pd.__version__,'; geopandas',gpd.__version__,'; plotly',plotly.__version__)


pandas 2.2.2 ; geopandas 1.0.1 ; plotly 5.24.1


In [3]:
#Chemin et dossiers

RACINE=Path(os.environ.get('GEOMARKETING_RACINE',r'C:\Users\almou\OneDrive\GeoMarketing_IDF'))
FICHIER_PROFIL=RACINE/'data/processed/j13c/profil_iris_idf_j13c.csv'
FICHIER_GEO=RACINE/'data/processed/j13c/profil_iris_idf_j13c.gpkg'
FICHIER_META_J13C=RACINE/'data/processed/j13c/metadata_affectations_iris_j13c.json'
COUCHE_IRIS='profils_iris'
DOSSIER_RAW=RACINE/'data/raw/insee/rp2022/mobilites'
DOSSIER_COG=RACINE/'data/raw/insee/cog2024'
DOSSIER_INTERIM=RACINE/'data/interim/j13d'
DOSSIER_PROCESSED=RACINE/'data/processed/j13d'
DOSSIER_CARTES=RACINE/'outputs/cartes/j13d'
DOSSIER_DOCS=RACINE/'docs'
for dossier in [DOSSIER_RAW,DOSSIER_COG,DOSSIER_INTERIM,DOSSIER_PROCESSED,DOSSIER_CARTES,DOSSIER_DOCS]:
    dossier.mkdir(parents=True,exist_ok=True)
print('Racine :',RACINE)
print('Entrées J13c :',[(p.name,p.is_file()) for p in [FICHIER_PROFIL,FICHIER_GEO,FICHIER_META_J13C]])



Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Entrées J13c : [('profil_iris_idf_j13c.csv', True), ('profil_iris_idf_j13c.gpkg', True), ('metadata_affectations_iris_j13c.json', True)]


In [4]:
#Téléchargement et configuration des sources

ANNEE_RP=2022
GEOGRAPHIE=2024
ANNEE_COG=2024  # À vérifier dans la notice du fichier réellement téléchargé.
DATE_TELECHARGEMENT_COG=None
URL_COG='https://www.insee.fr/fr/information/7766585'
TAILLE_BLOC=100_000
DEPARTEMENTS_IDF={'75','77','78','91','92','93','94','95'}
CODES_FOCUS=['95127','95500']
CRS_TRAVAIL=2154
AFFICHER_CARTES=False
TOP_LIENS_CARTE=30
TOP_PARTENAIRES=12
SIMPLIFICATION_METRES=20
CONFIG_PLOTLY={'scrollZoom':True,'displaylogo':False}
FICHIER_COG=DOSSIER_COG/'v_commune_2024.csv'
SOURCES={
 'MOBPRO':{'type_fichier':'DETAIL','date_telechargement':None,'actif':True,'fichier':None,'membre':None,'sep':None,'encoding':'utf-8-sig',
           'annee':2022,'geographie':2024,'origine':'COMMUNE','destination':'DCLT',
           'poids':'IPONDI','mode':'TRANS','age':None,'analyser_modes':True,'analyser_ages':False,
           'url':'https://www.insee.fr/fr/statistiques/8589904?sommaire=8590191',
           'champ':'Actifs ayant un emploi, âgés de 15 ans ou plus'},
 'MOBSCO':{'type_fichier':'DETAIL','date_telechargement':None,'actif':True,'fichier':None,'membre':None,'sep':None,'encoding':'utf-8-sig',
           'annee':2022,'geographie':2024,'origine':'COMMUNE','destination':'DCETUF',
           'poids':'IPONDI','mode':None,'age':'AGEREV10','analyser_modes':False,'analyser_ages':True,
           'url':'https://www.insee.fr/fr/statistiques/8589945?sommaire=8590191',
           'champ':'Personnes inscrites dans un établissement d’enseignement, âgées de 2 ans ou plus'},
}
def configurer_type_source(theme,type_fichier):
    if type_fichier not in {'DETAIL','FLUX'}:raise ValueError('Type attendu : DETAIL ou FLUX.')
    cfg=SOURCES[theme]
    cfg['type_fichier']=type_fichier
    if type_fichier=='DETAIL':
        cfg.update({'origine':'COMMUNE','destination':'DCLT' if theme=='MOBPRO' else 'DCETUF',
            'poids':'IPONDI','mode':'TRANS' if theme=='MOBPRO' else None,
            'age':'AGEREV10' if theme=='MOBSCO' else None,'analyser_modes':theme=='MOBPRO',
            'analyser_ages':theme=='MOBSCO','geographie':2024,
            'url':'https://www.insee.fr/fr/statistiques/'+('8589904' if theme=='MOBPRO' else '8589945')+'?sommaire=8590191'})
    else:
        cfg.update({'origine':'CODGEO','destination':'DCLT' if theme=='MOBPRO' else 'DCETU',
            'poids':'NBFLUX_C22_ACTOCC15P' if theme=='MOBPRO' else 'NBFLUX_C22_SCOL02P',
            'mode':None,'age':None,'analyser_modes':False,'analyser_ages':False,'geographie':2025,
            'url':'https://www.insee.fr/fr/statistiques/'+('8582949' if theme=='MOBPRO' else '8582969')})

# Parcours principal : DETAIL pour les deux sources. FLUX exige des entrées
# J13c et un COG compatibles avec sa géographie, contrôlés aux blocs 5 et 7.
# configurer_type_source('MOBPRO','FLUX')
# configurer_type_source('MOBSCO','FLUX')

# Exemples de chemin, à adapter au vrai nom du téléchargement :
# SOURCES['MOBPRO']['fichier']=DOSSIER_RAW/'votre_fichier_travail.csv'
# SOURCES['MOBSCO']['fichier']=DOSSIER_RAW/'votre_archive_etudes.zip'
if not any(c['actif'] for c in SOURCES.values()):raise ValueError('Activer au moins une source.')
for theme,cfg in SOURCES.items():
    if cfg['type_fichier'] not in {'DETAIL','FLUX'}:raise ValueError(theme+' : type_fichier non reconnu.')



In [5]:
#Fonctions pour les noms, les codes et lettres

def nom_colonne(nom):
    s=unicodedata.normalize('NFKD',str(nom).replace('\ufeff','').strip().upper())
    s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]+','_',s).strip('_')
def entetes(table):
    t=table.copy();t.columns=[nom_colonne(c) for c in t.columns]
    if t.columns.duplicated().any():raise ValueError('Noms de colonnes dupliqués après normalisation.')
    return t
def codes(serie):
    s=serie.astype('string').str.strip().str.upper().str.replace(r'\.0$','',regex=True)
    s=s.replace({'':pd.NA,'NAN':pd.NA,'NONE':pd.NA,'<NA>':pd.NA})
    masque=s.str.fullmatch(r'\d{1,5}',na=False)
    return s.where(~masque,s.str.zfill(5))
def nombre(serie):
    s=serie.astype('string').str.strip().str.replace('\u202f','',regex=False).str.replace('\u00a0','',regex=False)
    s=s.str.replace(' ','',regex=False).str.replace(',','.',regex=False)
    return pd.to_numeric(s,errors='coerce')
def exporter(table,chemin):
    table.to_csv(chemin,index=False,encoding='utf-8-sig',float_format='%.15g')
    return chemin
def empreinte(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for bloc in iter(lambda:f.read(1024*1024),b''):h.update(bloc)
    return h.hexdigest()
def requis_source(cfg):
    cols=[cfg['origine'],cfg['destination'],cfg['poids']]
    if cfg['analyser_modes']:cols.append(cfg['mode'])
    if cfg['analyser_ages']:cols.append(cfg['age'])
    if None in cols:raise ValueError('Configuration de colonne incomplète.')
    return list(dict.fromkeys(cols))
@contextmanager
def flux_binaire(fichier,membre=None):
    p=Path(fichier)
    if p.suffix.lower()=='.zip':
        with ZipFile(p) as z:
            with z.open(membre) as f:yield f
    else:
        with p.open('rb') as f:yield f
def entete_flux(fichier,membre,encoding,sep=None):
    with flux_binaire(fichier,membre) as f:ligne=f.readline(32768).decode(encoding)
    if not ligne.strip():raise ValueError('En-tête vide.')
    essais=[]
    for delim in ([sep] if sep else [';',',','\t']):
        cols=next(csv.reader([ligne],delimiter=delim))
        essais.append((len(cols),delim,cols))
    _,delim,cols=max(essais,key=lambda x:x[0])
    norm=[nom_colonne(c) for c in cols]
    if len(set(norm))!=len(norm):raise ValueError('En-tête ambigu après normalisation.')
    return {'sep':delim,'colonnes':norm,'brutes':cols}
def inspecter_csv(fichier,requis,membre=None,encoding='utf-8-sig',sep=None):
    p=Path(fichier)
    if not p.is_file():raise FileNotFoundError(str(p))
    if p.suffix.lower() not in {'.csv','.zip'}:raise ValueError('Utiliser un CSV ou ZIP contenant un CSV.')
    if p.suffix.lower()=='.zip':
        with ZipFile(p) as z:
            noms=[i.filename for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith('.csv')]
        if membre is not None:
            if membre not in noms:raise ValueError('Membre ZIP absent : '+str(membre))
            noms=[membre]
    else:noms=[None]
    valides=[];examines=[]
    for m in noms:
        info=entete_flux(p,m,encoding,sep);examines.append((m,info['colonnes']))
        if set(requis)<=set(info['colonnes']):valides.append({'fichier':p,'membre':m,'encoding':encoding,**info})
    if len(valides)!=1:
        raise ValueError('Choisir un CSV au schéma exact ou préciser membre. Candidats : '+str(examines))
    return valides[0]
def blocs_csv(info,colonnes,taille=TAILLE_BLOC):
    import io
    if taille<1:raise ValueError('TAILLE_BLOC doit être positive.')
    indices=[i for i,c in enumerate(info['colonnes']) if c in colonnes]
    noms=[info['colonnes'][i] for i in indices]
    with flux_binaire(info['fichier'],info['membre']) as f:
        lecteur=csv.reader(io.TextIOWrapper(f,encoding=info['encoding'],newline=''),
                           delimiter=info['sep'],strict=True)
        try:
            en_tete=[nom_colonne(c) for c in next(lecteur)]
            if en_tete!=info['colonnes']:raise ValueError('En-tête modifié entre inspection et lecture.')
            stock=[]
            for ligne in lecteur:
                if not ligne:continue
                if len(ligne)!=len(info['colonnes']):
                    raise pd.errors.ParserError('CSV mal formé à la ligne physique '+str(lecteur.line_num)+
                        ' : '+str(len(ligne))+' champs au lieu de '+str(len(info['colonnes']))+'.')
                stock.append([ligne[i] for i in indices])
                if len(stock)>=taille:
                    yield pd.DataFrame(stock,columns=noms,dtype='string');stock=[]
            if stock:yield pd.DataFrame(stock,columns=noms,dtype='string')
        except csv.Error as e:
            raise pd.errors.ParserError('CSV mal formé près de la ligne physique '+str(lecteur.line_num)+'. '+str(e)) from e
def lire_petit_csv(p):
    info=inspecter_csv(p,[])
    return pd.concat(list(blocs_csv(info,info['colonnes'])),ignore_index=True)



In [9]:
#Vérifier les fichiers

SOURCES['MOBPRO']['fichier']=DOSSIER_RAW/'RP2022_MOBPRO/FD_MOBPRO_2022.csv'
SOURCES['MOBSCO']['fichier']=DOSSIER_RAW/'RP2022_MOBSCO/FD_MOBSCO_2022.csv'

inventaire=[{'CHEMIN':str(p.relative_to(RACINE)),'TAILLE_MO':p.stat().st_size/1_000_000}
            for dossier in [DOSSIER_RAW,DOSSIER_COG] for p in sorted(dossier.rglob('*')) if p.is_file()]
exporter(pd.DataFrame(inventaire,columns=['CHEMIN','TAILLE_MO']),DOSSIER_INTERIM/'inventaire_j13d.csv')
if ANNEE_COG!=GEOGRAPHIE:raise ValueError('Le COG doit correspondre à GEOGRAPHIE='+str(GEOGRAPHIE)+'.')
LECTURES={}
for theme,cfg in SOURCES.items():
    if not cfg['actif']:continue
    if cfg['annee']!=ANNEE_RP or cfg['geographie']!=GEOGRAPHIE:
        raise ValueError(theme+' : la source '+cfg['type_fichier']+' est en géographie '+str(cfg['geographie'])+
            ' ; J13d est configuré en '+str(GEOGRAPHIE)+'. Utiliser les fichiers détail 2022/2024 pour le parcours principal, '+
            'ou fournir des entrées J13c et un COG explicitement harmonisés à la géographie choisie.')
    if cfg['fichier'] is None:raise ValueError(theme+' : renseigner le chemin exact dans le bloc 3.')
    info=inspecter_csv(cfg['fichier'],requis_source(cfg),cfg['membre'],cfg['encoding'],cfg['sep'])
    LECTURES[theme]=info
    exporter(pd.DataFrame({'COLONNE':info['colonnes']}),DOSSIER_INTERIM/f'entetes_{theme.lower()}_j13d.csv')
    print(theme,':',info['fichier'].name,'; membre :',info['membre'],'; séparateur :',repr(info['sep']))
for p in [FICHIER_PROFIL,FICHIER_GEO,FICHIER_META_J13C,FICHIER_COG]:
    if not p.is_file():raise FileNotFoundError('Entrée requise : '+str(p))



MOBPRO : FD_MOBPRO_2022.csv ; membre : None ; séparateur : ';'
MOBSCO : FD_MOBSCO_2022.csv ; membre : None ; séparateur : ';'


In [10]:
cog=entetes(lire_petit_csv(FICHIER_COG))
requis={'TYPECOM','COM','COMPARENT','REG','DEP','LIBELLE'}
if requis-set(cog):raise ValueError('COG : colonnes absentes '+str(sorted(requis-set(cog))))
cog['COM']=codes(cog['COM']);cog['COMPARENT']=codes(cog['COMPARENT'])
cog['REG']=cog['REG'].str.strip().str.zfill(2);cog['DEP']=cog['DEP'].str.strip()
if cog[['TYPECOM','COM']].duplicated().any():raise ValueError('COG : clés dupliquées.')
communes=cog[cog['TYPECOM'].eq('COM')][['COM','LIBELLE','REG','DEP']].copy().rename(columns={'COM':'CODGEO','LIBELLE':'NOM_COMMUNE'})
if not communes['CODGEO'].is_unique or communes['CODGEO'].isna().any():raise ValueError('COG COM : codes invalides.')
ref=communes.set_index('CODGEO')
arr=cog[cog['TYPECOM'].eq('ARM')][['COM','COMPARENT']].copy()
if not arr['COM'].is_unique or not arr['COMPARENT'].isin(ref.index).all():raise ValueError('COG : parents ARM invalides.')
PARENTS_ARM=arr.set_index('COM')['COMPARENT'].to_dict()
def canonicaliser(serie):
    s=codes(serie)
    return s.map(PARENTS_ARM).fillna(s)
CODES_FRANCE=set(ref.index)
CODES_IDF=set(communes.loc[communes['REG'].eq('11'),'CODGEO'])
if set(ref.loc[sorted(CODES_IDF),'DEP'])!=DEPARTEMENTS_IDF:raise ValueError('COG : les huit départements IDF sont requis.')
FOCUS=set(codes(pd.Series(CODES_FOCUS,dtype='string')))
if not FOCUS or FOCUS-CODES_IDF:raise ValueError('Focus : sélectionner des communes COM du COG 2024 en IDF.')
exporter(arr,DOSSIER_INTERIM/'passage_arrondissements_communes_2024.csv')
print('Communes IDF attendues :',len(CODES_IDF),' ; focus :',sorted(FOCUS))


Communes IDF attendues : 1267  ; focus : ['95127', '95500']


In [11]:
#Reprendre tous les IRIS J13c

meta_j13c=json.loads(FICHIER_META_J13C.read_text(encoding='utf-8'))
if meta_j13c.get('annee_rp')!=ANNEE_RP or meta_j13c.get('geographie_iris')!=GEOGRAPHIE:
    raise ValueError('Métadonnées J13c incompatibles : vérifier millésime et géographie.')
base=entetes(lire_petit_csv(FICHIER_PROFIL))
obligatoires={'IRIS','CODGEO_SOURCE','CODGEO_ETUDE','J13A_POPULATION','J13A_ANNEE_RP','J13A_GEOGRAPHIE'}
if obligatoires-set(base):raise ValueError('Profil J13c incomplet : '+str(sorted(obligatoires-set(base))))
if any(c.startswith('J13D_') for c in base):raise ValueError('Choisir J13c comme entrée, pas une ancienne sortie J13d.')
base['IRIS']=base['IRIS'].str.strip()
if not base['IRIS'].str.fullmatch(r'\d{9}',na=False).all() or not base['IRIS'].is_unique:raise ValueError('IRIS invalides ou dupliqués.')
if not nombre(base['J13A_ANNEE_RP']).eq(ANNEE_RP).all() or not nombre(base['J13A_GEOGRAPHIE']).eq(GEOGRAPHIE).all():
    raise ValueError('Profil J13c : RP/géographie différents.')
code_source=codes(base['CODGEO_SOURCE'])
if not code_source.eq(base['IRIS'].str[:5]).all():raise ValueError('IRIS / commune source incohérents.')
cle_commune=canonicaliser(code_source)
if set(cle_commune)!=CODES_IDF:
    raise ValueError('Couverture J13c / COG différente : absentes '+str(sorted(CODES_IDF-set(cle_commune)))+
                     ' ; supplémentaires '+str(sorted(set(cle_commune)-CODES_IDF)))
iris=gpd.read_file(FICHIER_GEO,layer=COUCHE_IRIS)
if 'IRIS' not in iris or iris.crs is None:raise ValueError('Contours : IRIS et CRS requis.')
iris['IRIS']=iris['IRIS'].astype('string').str.strip()
if not iris['IRIS'].is_unique or set(iris['IRIS'])!=set(base['IRIS']):raise ValueError('Clés CSV/GPKG différentes.')
if iris.geometry.isna().any() or iris.geometry.is_empty.any() or not iris.geometry.is_valid.all():raise ValueError('Contours vides ou invalides.')
if not iris.geom_type.isin(['Polygon','MultiPolygon']).all():raise ValueError('Des polygones sont requis.')
iris=iris[['IRIS',iris.geometry.name]].to_crs(CRS_TRAVAIL)
if iris.geometry.name!='geometry':iris=iris.rename_geometry('geometry')
geo_communes=iris.merge(pd.DataFrame({'IRIS':base['IRIS'],'CODGEO':cle_commune}),on='IRIS',validate='one_to_one')
geo_communes=geo_communes.dissolve(by='CODGEO',as_index=False)[['CODGEO','geometry']]
communes_idf=ref.loc[sorted(CODES_IDF)].reset_index()
pop=nombre(base['J13A_POPULATION'])
if pop.lt(0).any():raise ValueError('Population négative dans J13c.')
pop_commune=pd.DataFrame({'CODGEO':cle_commune,'POP':pop}).groupby('CODGEO')['POP'].agg(lambda s:s.sum() if s.notna().all() else np.nan)
communes_idf['J13D_POPULATION_RP_IRIS']=communes_idf['CODGEO'].map(pop_commune)
communes_idf['J13D_NB_IRIS']=communes_idf['CODGEO'].map(cle_commune.value_counts())
ATTENDUS=set(base['IRIS']);NOMBRE_IRIS=len(base)
print('IRIS conservés :',NOMBRE_IRIS,' ; communes :',len(communes_idf))


IRIS conservés : 5264  ; communes : 1267


In [12]:
#Lecture progressive

COLS_OD=['THEME','ORIGINE','DESTINATION','STATUT_DESTINATION','MODE_CODE','AGE_CODE']
MODES={'1':'Sans transport','2':'Marche','3':'Vélo','4':'Deux-roues motorisé',
       '5':'Voiture, camion ou fourgonnette','6':'Transports en commun','ND':'Non renseigné'}
AGES={'02':'2 ans','03':'3 ans','04':'4 ans','05':'5 ans','06':'6–10 ans','11':'11–14 ans',
      '15':'15–17 ans','18':'18–24 ans','25':'25–29 ans','30':'30 ans ou plus','ND':'Non renseigné'}
MODALITES_A_CONTROLER=[]
def traiter_source(theme,cfg,info):
    total=defaultdict(lambda:[0.0,0]);qa=[];n_lus=0;n_retenus=0;w_retenu=0.0
    poids_mode_nd=0.0;poids_age_nd=0.0;cles_flux_vues=set()
    for numero,b in enumerate(blocs_csv(info,requis_source(cfg)),start=1):
        n_lus+=len(b)
        o=canonicaliser(b[cfg['origine']]);d=canonicaliser(b[cfg['destination']])
        hint_o=codes(b[cfg['origine']]).str[:2].isin(DEPARTEMENTS_IDF)
        hint_d=codes(b[cfg['destination']]).str[:2].isin(DEPARTEMENTS_IDF)
        keep=o.isin(CODES_IDF)|d.isin(CODES_IDF)|hint_o|hint_d
        b=b.loc[keep].copy();o=o.loc[keep];d=d.loc[keep]
        if b.empty:continue
        w=nombre(b[cfg['poids']]).astype('float64')
        mauvais=~np.isfinite(w)|(w.le(0) if cfg['type_fichier']=='DETAIL' else w.lt(0))
        if mauvais.any():
            exporter(b.loc[mauvais].head(100),DOSSIER_INTERIM/f'erreur_poids_{theme.lower()}.csv')
            raise ValueError(theme+' : valeurs/poids invalides. Voir le diagnostic ; aucune ligne ignorée.')
        if cfg['type_fichier']=='FLUX':
            cles=list(zip(codes(b[cfg['origine']]).fillna('NON_LOCALISE'),codes(b[cfg['destination']]).fillna('NON_LOCALISE')))
            doubles=[]
            for cle in cles:
                doubles.append(cle in cles_flux_vues);cles_flux_vues.add(cle)
            if any(doubles):
                exporter(b.loc[pd.Series(doubles,index=b.index)].head(100),DOSSIER_INTERIM/f'doublons_flux_{theme.lower()}.csv')
                raise ValueError(theme+' : paire origine-destination répétée dans la table FLUX ; vérifier la source, aucun double comptage accepté.')
        mauvais_orig=~o.isin(CODES_FRANCE)
        mauvais_idf=hint_d.loc[keep]&~d.isin(CODES_IDF)
        if mauvais_orig.any() or mauvais_idf.any():
            exporter(b.loc[mauvais_orig|mauvais_idf].head(100),DOSSIER_INTERIM/f'erreur_codes_{theme.lower()}.csv')
            raise ValueError(theme+' : code origine ou destination IDF non reconnu par le COG 2024.')
        statut=pd.Series('NON_REFERENCEE',index=b.index,dtype='string')
        statut.loc[d.isin(CODES_FRANCE)]='COMMUNE_REFERENCEE'
        statut.loc[d.eq('99999').fillna(False)]='ETRANGER'
        statut.loc[d.isna()|d.isin(['YYYYY','ZZZZZ'])]='NON_LOCALISEE'
        dest=d.fillna('NON_LOCALISE')
        dest.loc[statut.eq('ETRANGER')]='ETRANGER'
        dest.loc[statut.eq('NON_LOCALISEE')]='NON_LOCALISE'
        mode=pd.Series('NA',index=b.index,dtype='string')
        age=pd.Series('NA',index=b.index,dtype='string')
        if cfg['analyser_modes']:
            m=b[cfg['mode']].str.strip().str.upper().replace({'':'ND','Z':'ND'})
            m=m.where(m.isin(MODES),'ND');mode=m
            poids_mode_nd+=float(w.loc[m.eq('ND')].sum())
        if cfg['analyser_ages']:
            a=b[cfg['age']].str.strip().str.upper().str.zfill(2)
            a=a.where(a.isin(AGES),'ND');age=a
            poids_age_nd+=float(w.loc[a.eq('ND')].sum())
        for colonne,modalite,active in [(cfg['mode'],mode,cfg['analyser_modes']),
                                        (cfg['age'],age,cfg['analyser_ages'])]:
            if not active or not modalite.eq('ND').any():continue
            diagnostic=pd.DataFrame({'VALEUR_SOURCE':b.loc[modalite.eq('ND'),colonne],
                                     'POIDS':w.loc[modalite.eq('ND')]}).groupby('VALEUR_SOURCE',dropna=False)['POIDS'].agg(['sum','size']).reset_index()
            for r in diagnostic.itertuples(index=False):
                MODALITES_A_CONTROLER.append({'THEME':theme,'VARIABLE':colonne,'VALEUR_SOURCE':r.VALEUR_SOURCE,
                                              'EFFECTIF_PONDERE':float(r.sum),'NB_ENREGISTREMENTS':int(r.size)})
        t=pd.DataFrame({'THEME':theme,'ORIGINE':o,'DESTINATION':dest,'STATUT_DESTINATION':statut,
                        'MODE_CODE':mode,'AGE_CODE':age,'EFFECTIF_PONDERE':w,'NB_ENREGISTREMENTS':1})
        ag=t.groupby(COLS_OD,dropna=False)[['EFFECTIF_PONDERE','NB_ENREGISTREMENTS']].sum().reset_index()
        for row in ag.itertuples(index=False,name=None):
            key=row[:len(COLS_OD)];total[key][0]+=float(row[-2]);total[key][1]+=int(row[-1])
        n_retenus+=len(t);w_retenu+=float(w.sum())
        qa.append({'THEME':theme,'BLOC':numero,'NB_LIGNES_RETENUES':len(t),'EFFECTIF_PONDERE':float(w.sum())})
        if numero==1 or numero%10==0:print(theme,': blocs',numero,'; lignes lues',n_lus,'; retenues',n_retenus)
    table=pd.DataFrame([dict(zip(COLS_OD,k),EFFECTIF_PONDERE=v[0],NB_ENREGISTREMENTS=v[1]) for k,v in total.items()],
                       columns=COLS_OD+['EFFECTIF_PONDERE','NB_ENREGISTREMENTS'])
    table['EFFECTIF_PONDERE']=pd.to_numeric(table['EFFECTIF_PONDERE']).astype(float)
    table['NB_ENREGISTREMENTS']=pd.to_numeric(table['NB_ENREGISTREMENTS']).astype('int64')
    if not np.isclose(table['EFFECTIF_PONDERE'].sum(),w_retenu,rtol=1e-10,atol=1e-8):raise AssertionError('Poids perdus à l’agrégation.')
    if int(table['NB_ENREGISTREMENTS'].sum())!=n_retenus:raise AssertionError('Enregistrements perdus.')
    qualite={'TYPE_FICHIER':cfg['type_fichier'],'COLONNE_EFFECTIF':cfg['poids'],'THEME':theme,'NB_LIGNES_LUES':n_lus,'NB_LIGNES_RETENUES':n_retenus,
             'EFFECTIF_PONDERE_RETENU':w_retenu,'NB_GROUPES_DETAIL':len(table),
             'MODES_DISPONIBLES':cfg['analyser_modes'],'AGES_DISPONIBLES':cfg['analyser_ages'],
             'POIDS_MODE_NON_RENSEIGNE':poids_mode_nd if cfg['analyser_modes'] else None,
             'POIDS_AGE_NON_RENSEIGNE':poids_age_nd if cfg['analyser_ages'] else None}
    return table,qualite,pd.DataFrame(qa,columns=['THEME','BLOC','NB_LIGNES_RETENUES','EFFECTIF_PONDERE'])
DETAILS={};QUALITE=[];JOURNAUX=[]
for theme,info in LECTURES.items():
    table,q,journal=traiter_source(theme,SOURCES[theme],info)
    DETAILS[theme]=table;QUALITE.append(q);JOURNAUX.append(journal)
detail=pd.concat(list(DETAILS.values()),ignore_index=True)
qualite=pd.DataFrame(QUALITE)
exporter(qualite,DOSSIER_INTERIM/'qualite_sources_j13d.csv')
exporter(pd.concat(JOURNAUX,ignore_index=True),DOSSIER_INTERIM/'lecture_par_bloc_j13d.csv')
modalites_nd=pd.DataFrame(MODALITES_A_CONTROLER,columns=['THEME','VARIABLE','VALEUR_SOURCE','EFFECTIF_PONDERE','NB_ENREGISTREMENTS'])
if not modalites_nd.empty:
    modalites_nd=modalites_nd.groupby(['THEME','VARIABLE','VALEUR_SOURCE'],dropna=False,as_index=False)[['EFFECTIF_PONDERE','NB_ENREGISTREMENTS']].sum()
exporter(modalites_nd,DOSSIER_INTERIM/'modalites_non_renseignees_j13d.csv')
display(qualite)


MOBPRO : blocs 1 ; lignes lues 100000 ; retenues 2821
MOBPRO : blocs 10 ; lignes lues 1000000 ; retenues 12880
MOBPRO : blocs 20 ; lignes lues 2000000 ; retenues 36321
MOBPRO : blocs 30 ; lignes lues 3000000 ; retenues 50620
MOBPRO : blocs 40 ; lignes lues 4000000 ; retenues 83227
MOBPRO : blocs 50 ; lignes lues 5000000 ; retenues 258325
MOBPRO : blocs 60 ; lignes lues 6000000 ; retenues 790414
MOBPRO : blocs 70 ; lignes lues 7000000 ; retenues 1595051
MOBSCO : blocs 1 ; lignes lues 100000 ; retenues 615
MOBSCO : blocs 10 ; lignes lues 1000000 ; retenues 5972
MOBSCO : blocs 20 ; lignes lues 2000000 ; retenues 10676
MOBSCO : blocs 30 ; lignes lues 3000000 ; retenues 259231
MOBSCO : blocs 40 ; lignes lues 4000000 ; retenues 968192


,TYPE_FICHIER,COLONNE_EFFECTIF,THEME,NB_LIGNES_LUES,NB_LIGNES_RETENUES,EFFECTIF_PONDERE_RETENU,NB_GROUPES_DETAIL,MODES_DISPONIBLES,AGES_DISPONIBLES,POIDS_MODE_NON_RENSEIGNE,POIDS_AGE_NON_RENSEIGNE
0,DETAIL,IPONDI,MOBPRO,7388023,1786201,6.090445e+06,235377,True,False,0.0,NaN
1,DETAIL,IPONDI,MOBSCO,4169977,986546,3.302615e+06,94003,False,True,NaN,0.0


In [13]:
#Construire les matrices 

flux=detail.groupby(['THEME','ORIGINE','DESTINATION','STATUT_DESTINATION'],as_index=False)[['EFFECTIF_PONDERE','NB_ENREGISTREMENTS']].sum()
flux['ORDRE_DE_GRANDEUR_MOINS_200']=flux['EFFECTIF_PONDERE'].lt(200)
flux['ORIGINE_IDF']=flux['ORIGINE'].isin(CODES_IDF)
flux['DESTINATION_IDF']=flux['DESTINATION'].isin(CODES_IDF)
flux['NOM_ORIGINE']=flux['ORIGINE'].map(ref['NOM_COMMUNE'])
flux['NOM_DESTINATION']=flux['DESTINATION'].map(ref['NOM_COMMUNE'])
flux.loc[flux['STATUT_DESTINATION'].eq('ETRANGER'),'NOM_DESTINATION']='Étranger (commune non localisée ici)'
flux.loc[flux['STATUT_DESTINATION'].eq('NON_LOCALISEE'),'NOM_DESTINATION']='Destination non localisée'
inconnue=flux['STATUT_DESTINATION'].eq('NON_REFERENCEE')
flux.loc[inconnue,'NOM_DESTINATION']='Code non référencé : '+flux.loc[inconnue,'DESTINATION'].astype('string')
flux['REGION_ORIGINE']=flux['ORIGINE'].map(ref['REG'])
flux['REGION_DESTINATION']=flux['DESTINATION'].map(ref['REG'])
flux=flux.sort_values(['THEME','EFFECTIF_PONDERE'],ascending=[True,False]).reset_index(drop=True)
assert (flux['ORIGINE_IDF']|flux['DESTINATION_IDF']).all()
assert not flux[['THEME','ORIGINE','DESTINATION','STATUT_DESTINATION']].duplicated().any()
display(flux.head(15))


,THEME,ORIGINE,DESTINATION,STATUT_DESTINATION,EFFECTIF_PONDERE,NB_ENREGISTREMENTS,ORDRE_DE_GRANDEUR_MOINS_200,ORIGINE_IDF,DESTINATION_IDF,NOM_ORIGINE,NOM_DESTINATION,REGION_ORIGINE,REGION_DESTINATION
0,MOBPRO,75056,75056,COMMUNE_REFERENCEE,752603.145072,235277,False,True,True,Paris,Paris,11,11
1,MOBPRO,92012,75056,COMMUNE_REFERENCEE,21596.284201,6727,False,True,True,Boulogne-Billancourt,Paris,11,11
2,MOBPRO,93048,75056,COMMUNE_REFERENCEE,19558.442847,6349,False,True,True,Montreuil,Paris,11,11
3,MOBPRO,75056,92026,COMMUNE_REFERENCEE,19480.536079,6123,False,True,True,Paris,Courbevoie,11,11
4,MOBPRO,92012,92012,COMMUNE_REFERENCEE,17901.150603,5606,False,True,True,Boulogne-Billancourt,Boulogne-Billancourt,11,11
5,MOBPRO,75056,92062,COMMUNE_REFERENCEE,17122.703587,5328,False,True,True,Paris,Puteaux,11,11
6,MOBPRO,75056,92012,COMMUNE_REFERENCEE,16408.203780,5059,False,True,True,Paris,Boulogne-Billancourt,11,11
7,MOBPRO,92004,75056,COMMUNE_REFERENCEE,16282.217011,5295,False,True,True,Asnières-sur-Seine,Paris,11,11
8,MOBPRO,93066,75056,COMMUNE_REFERENCEE,14687.019556,4711,False,True,True,Saint-Denis,Paris,11,11
9,MOBPRO,92050,92050,COMMUNE_REFERENCEE,14375.543766,4431,False,True,True,Nanterre,Nanterre,11,11


In [14]:
#Controler les destinations et les échanges hors IDF

diagnostic_dest=flux[~flux['STATUT_DESTINATION'].eq('COMMUNE_REFERENCEE')].copy()
exporter(diagnostic_dest,DOSSIER_INTERIM/'destinations_a_documenter_j13d.csv')
couverture_dest=flux.groupby(['THEME','STATUT_DESTINATION'],as_index=False)['EFFECTIF_PONDERE'].sum()
echange=[]
for theme,t in flux.groupby('THEME'):
    masques={'IDF_VERS_IDF':t['ORIGINE_IDF']&t['DESTINATION_IDF'],
             'HORS_IDF_VERS_IDF':~t['ORIGINE_IDF']&t['DESTINATION_IDF'],
             'IDF_VERS_FRANCE_HORS_IDF':t['ORIGINE_IDF']&~t['DESTINATION_IDF']&t['STATUT_DESTINATION'].eq('COMMUNE_REFERENCEE'),
             'IDF_VERS_ETRANGER':t['ORIGINE_IDF']&t['STATUT_DESTINATION'].eq('ETRANGER'),
             'IDF_VERS_DEST_NON_REFERENCEE_OU_NON_LOCALISEE':t['ORIGINE_IDF']&t['STATUT_DESTINATION'].isin(['NON_REFERENCEE','NON_LOCALISEE'])}
    for nom,m in masques.items():echange.append({'THEME':theme,'ECHANGE':nom,'EFFECTIF_PONDERE':float(t.loc[m,'EFFECTIF_PONDERE'].sum())})
echanges=pd.DataFrame(echange)
exporter(couverture_dest,DOSSIER_INTERIM/'couverture_destinations_j13d.csv')
exporter(echanges,DOSSIER_INTERIM/'echanges_idf_hors_idf_j13d.csv')
display(echanges)


,THEME,ECHANGE,EFFECTIF_PONDERE
0,MOBPRO,IDF_VERS_IDF,5.581382e+06
1,MOBPRO,HORS_IDF_VERS_IDF,4.130982e+05
2,MOBPRO,IDF_VERS_FRANCE_HORS_IDF,8.698834e+04
3,MOBPRO,IDF_VERS_ETRANGER,8.883707e+03
4,MOBPRO,IDF_VERS_DEST_NON_REFERENCEE_OU_NON_LOCALISEE,9.312425e+01
5,MOBSCO,IDF_VERS_IDF,3.165652e+06
6,MOBSCO,HORS_IDF_VERS_IDF,7.113543e+04
7,MOBSCO,IDF_VERS_FRANCE_HORS_IDF,6.536834e+04
8,MOBSCO,IDF_VERS_ETRANGER,4.584969e+02
9,MOBSCO,IDF_VERS_DEST_NON_REFERENCEE_OU_NON_LOCALISEE,1.000063e+00


In [15]:
#Calculer les indicateurs communaux

DICTIONNAIRE=[]
def decrire(col,libelle,unite,formule,theme='COMMUNES',agregation='SOMME_COMMUNES'):
    DICTIONNAIRE[:]=[d for d in DICTIONNAIRE if d['COLONNE']!=col]
    DICTIONNAIRE.append({'COLONNE':col,'LIBELLE':libelle,'UNITE':unite,'FORMULE':formule,
                        'THEME':theme,'ANNEE_RP':ANNEE_RP,'GEOGRAPHIE':GEOGRAPHIE,
                        'MAILLE':'COMMUNE','AGREGATION':agregation})
def ratio(a,b,facteur=100):return a.div(b.where(b.gt(0)))*facteur
def valeurs_groupe(t,masque,cle):return t.loc[masque].groupby(cle)['EFFECTIF_PONDERE'].sum()
indicateurs=communes_idf.copy()
LIBELLES_MESURES={
 'RESIDENTS':'Personnes du champ résidant dans la commune',
 'INTERNES':'Résidence et lieu déclaré dans la même commune',
 'SORTANTS_FRANCE':'Résidents dont le lieu déclaré est dans une autre commune française référencée',
 'SORTANTS_ETRANGER':'Résidents dont le lieu déclaré est à l’étranger',
 'DEST_NON_LOCALISEES':'Résidents dont la destination est non localisée ou non référencée',
 'ENTRANTS':'Personnes résidant ailleurs et déclarant leur lieu dans la commune',
 'LIEU':'Personnes déclarant leur lieu dans la commune',
 'SOLDE_FRANCE':'Entrants moins sortants vers les communes françaises référencées',
 'PART_INTERNES_DEST_CONNUE_PCT':'Part des internes parmi les résidents à destination communale connue',
 'PART_ENTRANTS_LIEU_PCT':'Part des entrants parmi les personnes au lieu déclaré',
 'PART_DEST_NON_LOCALISEES_PCT':'Part des résidents à destination non localisée ou non référencée',
 'RATIO_LIEU_RESIDENTS':'Ratio personnes au lieu déclaré / personnes résidentes du même champ'}
MESURES=['RESIDENTS','INTERNES','SORTANTS_FRANCE','SORTANTS_ETRANGER','DEST_NON_LOCALISEES',
         'ENTRANTS','LIEU','SOLDE_FRANCE','PART_INTERNES_DEST_CONNUE_PCT','PART_ENTRANTS_LIEU_PCT',
         'PART_DEST_NON_LOCALISEES_PCT','RATIO_LIEU_RESIDENTS']
def calculer_communes(theme,t):
    tab=pd.DataFrame(index=sorted(CODES_IDF))
    connu=t['STATUT_DESTINATION'].eq('COMMUNE_REFERENCEE')
    interne=connu&t['ORIGINE'].eq(t['DESTINATION'])
    groupes={
       'RESIDENTS':valeurs_groupe(t,pd.Series(True,index=t.index),'ORIGINE'),
       'INTERNES':valeurs_groupe(t,interne,'ORIGINE'),
       'SORTANTS_FRANCE':valeurs_groupe(t,connu&~interne,'ORIGINE'),
       'SORTANTS_ETRANGER':valeurs_groupe(t,t['STATUT_DESTINATION'].eq('ETRANGER'),'ORIGINE'),
       'DEST_NON_LOCALISEES':valeurs_groupe(t,t['STATUT_DESTINATION'].isin(['NON_LOCALISEE','NON_REFERENCEE']),'ORIGINE'),
       'ENTRANTS':valeurs_groupe(t,connu&~interne,'DESTINATION'),
       'LIEU':valeurs_groupe(t,connu,'DESTINATION')}
    for m,s in groupes.items():tab[m]=s.reindex(tab.index,fill_value=0).astype(float)
    tab['SOLDE_FRANCE']=tab['ENTRANTS']-tab['SORTANTS_FRANCE']
    tab['PART_INTERNES_DEST_CONNUE_PCT']=ratio(tab['INTERNES'],tab['INTERNES']+tab['SORTANTS_FRANCE'])
    tab['PART_ENTRANTS_LIEU_PCT']=ratio(tab['ENTRANTS'],tab['LIEU'])
    tab['PART_DEST_NON_LOCALISEES_PCT']=ratio(tab['DEST_NON_LOCALISEES'],tab['RESIDENTS'])
    tab['RATIO_LIEU_RESIDENTS']=ratio(tab['LIEU'],tab['RESIDENTS'],1)
    return tab
for theme in SOURCES:
    charge=theme in DETAILS
    indicateurs[f'J13D_{theme}_CHARGE']=charge
    tab=calculer_communes(theme,flux[flux['THEME'].eq(theme)]) if charge else None
    for mesure in MESURES:
        col=f'J13D_{theme}_{mesure}'
        indicateurs[col]=indicateurs['CODGEO'].map(tab[mesure]) if charge else np.nan
        unite='%' if mesure.endswith('_PCT') else ('ratio' if mesure.startswith('RATIO_') else 'personnes pondérées')
        lib_theme='Travail — actifs occupés de 15 ans ou plus' if theme=='MOBPRO' else 'Études — personnes scolarisées de 2 ans ou plus'
        decrire(col,lib_theme+' : '+LIBELLES_MESURES[mesure],unite,
                {'SOLDE_FRANCE':'ENTRANTS - SORTANTS_FRANCE','LIEU':'INTERNES + ENTRANTS',
                 'RESIDENTS':'INTERNES + SORTANTS_FRANCE + SORTANTS_ETRANGER + DEST_NON_LOCALISEES',
                 'PART_INTERNES_DEST_CONNUE_PCT':'100 * INTERNES / (INTERNES + SORTANTS_FRANCE)',
                 'PART_ENTRANTS_LIEU_PCT':'100 * ENTRANTS / LIEU',
                 'PART_DEST_NON_LOCALISEES_PCT':'100 * DEST_NON_LOCALISEES / RESIDENTS',
                 'RATIO_LIEU_RESIDENTS':'LIEU / RESIDENTS'}.get(mesure,'Somme '+SOURCES[theme]['poids']+' du champ '+mesure),theme,
                'RECALCULER' if unite in {'%','ratio'} else 'SOMME_COMMUNES_SOUS_RESERVE_DU_PERIMETRE')
indicateurs['J13D_ANNEE_RP']=ANNEE_RP;indicateurs['J13D_GEOGRAPHIE']=GEOGRAPHIE
display(indicateurs.head())


,CODGEO,NOM_COMMUNE,REG,DEP,J13D_POPULATION_RP_IRIS,J13D_NB_IRIS,J13D_MOBPRO_CHARGE,J13D_MOBPRO_RESIDENTS,J13D_MOBPRO_INTERNES,J13D_MOBPRO_SORTANTS_FRANCE,J13D_MOBPRO_SORTANTS_ETRANGER,J13D_MOBPRO_DEST_NON_LOCALISEES,J13D_MOBPRO_ENTRANTS,J13D_MOBPRO_LIEU,J13D_MOBPRO_SOLDE_FRANCE,J13D_MOBPRO_PART_INTERNES_DEST_CONNUE_PCT,J13D_MOBPRO_PART_ENTRANTS_LIEU_PCT,J13D_MOBPRO_PART_DEST_NON_LOCALISEES_PCT,J13D_MOBPRO_RATIO_LIEU_RESIDENTS,J13D_MOBSCO_CHARGE,J13D_MOBSCO_RESIDENTS,J13D_MOBSCO_INTERNES,J13D_MOBSCO_SORTANTS_FRANCE,J13D_MOBSCO_SORTANTS_ETRANGER,J13D_MOBSCO_DEST_NON_LOCALISEES,J13D_MOBSCO_ENTRANTS,J13D_MOBSCO_LIEU,J13D_MOBSCO_SOLDE_FRANCE,J13D_MOBSCO_PART_INTERNES_DEST_CONNUE_PCT,J13D_MOBSCO_PART_ENTRANTS_LIEU_PCT,J13D_MOBSCO_PART_DEST_NON_LOCALISEES_PCT,J13D_MOBSCO_RATIO_LIEU_RESIDENTS,J13D_ANNEE_RP,J13D_GEOGRAPHIE
0,75056,Paris,11,75,2113705.0,992,True,1.077002e+06,752603.145072,321135.876736,3237.407949,25.999771,1.163508e+06,1.916111e+06,842371.811123,70.091813,60.722358,0.002414,1.779115,True,506478.139424,434802.671486,71490.201614,185.266324,0.0,307164.610601,741967.282088,235674.408988,85.879675,41.398673,0.0,1.464954,2022,2024
1,77001,Achères-la-Forêt,11,77,1183.0,1,True,6.219916e+02,88.819948,533.171628,0.000000,0.000000,1.163352e+02,2.051552e+02,-416.836385,14.279928,56.705971,0.000000,0.329836,True,220.736374,82.007178,138.729196,0.000000,0.0,33.656362,115.663540,-105.072834,37.151638,29.098506,0.0,0.523989,2022,2024
2,77002,Amillis,11,77,821.0,1,True,3.429200e+02,98.069386,244.850568,0.000000,0.000000,1.776356e+02,2.757050e+02,-67.214953,28.598332,64.429595,0.000000,0.803992,True,146.947501,62.105129,84.842372,0.000000,0.0,38.266499,100.371628,-46.575873,42.263481,38.124817,0.0,0.683044,2022,2024
3,77003,Amponville,11,77,351.0,1,True,1.586712e+02,38.465753,120.205479,0.000000,0.000000,1.004351e+01,4.850926e+01,-110.161973,24.242424,20.704308,0.000000,0.305722,True,57.698630,4.808219,52.890411,0.000000,0.0,4.794166,9.602385,-48.096245,8.333333,49.926822,0.0,0.166423,2022,2024
4,77004,Andrezel,11,77,324.0,1,True,1.645714e+02,36.000000,128.571429,0.000000,0.000000,1.484689e+01,5.084689e+01,-113.724536,21.875000,29.199213,0.000000,0.308965,True,66.857143,5.142857,61.714286,0.000000,0.0,5.047299,10.190156,-56.666987,7.692308,49.531124,0.0,0.152417,2022,2024


In [16]:
#Modes de transport des actifs occupés

def table_categories(theme,col,libelles,role_destination):
    colonnes=['THEME','ROLE','CODGEO','CATEGORIE_CODE','CATEGORIE','EFFECTIF_PONDERE','TOTAL_ROLE_COMMUNE','PART_PCT']
    cfg=SOURCES[theme]
    disponible=theme in DETAILS and (cfg['analyser_modes'] if col=='MODE_CODE' else cfg['analyser_ages'])
    if not disponible:return pd.DataFrame(columns=colonnes)
    t=DETAILS[theme];sorties=[]
    for role,cle in [('RESIDENCE','ORIGINE'),(role_destination,'DESTINATION')]:
        a=t[t[cle].isin(CODES_IDF)].groupby([cle,col],as_index=False)['EFFECTIF_PONDERE'].sum()
        a=a.rename(columns={cle:'CODGEO',col:'CATEGORIE_CODE'})
        a['THEME']=theme;a['ROLE']=role;a['CATEGORIE']=a['CATEGORIE_CODE'].map(libelles)
        a['TOTAL_ROLE_COMMUNE']=a.groupby('CODGEO')['EFFECTIF_PONDERE'].transform('sum')
        a['PART_PCT']=ratio(a['EFFECTIF_PONDERE'],a['TOTAL_ROLE_COMMUNE'])
        sorties.append(a[colonnes])
    return pd.concat(sorties,ignore_index=True)
modes=table_categories('MOBPRO','MODE_CODE',MODES,'LIEU_TRAVAIL')
exporter(modes,DOSSIER_PROCESSED/'modes_domicile_travail_communes_j13d.csv')
display(modes[modes['CODGEO'].isin(FOCUS)])


,THEME,ROLE,CODGEO,CATEGORIE_CODE,CATEGORIE,EFFECTIF_PONDERE,TOTAL_ROLE_COMMUNE,PART_PCT
5436,MOBPRO,RESIDENCE,95127,1,Sans transport,747.803914,28449.353783,2.628544
5437,MOBPRO,RESIDENCE,95127,2,Marche,1692.607541,28449.353783,5.949547
5438,MOBPRO,RESIDENCE,95127,3,Vélo,337.086351,28449.353783,1.184865
5439,MOBPRO,RESIDENCE,95127,4,Deux-roues motorisé,218.008873,28449.353783,0.766305
5440,MOBPRO,RESIDENCE,95127,5,"Voiture, camion ou fourgonnette",11084.108464,28449.353783,38.960844
5441,MOBPRO,RESIDENCE,95127,6,Transports en commun,14369.738639,28449.353783,50.509895
5915,MOBPRO,RESIDENCE,95500,1,Sans transport,395.360099,13873.881187,2.849672
5916,MOBPRO,RESIDENCE,95500,2,Marche,1369.565074,13873.881187,9.871535
5917,MOBPRO,RESIDENCE,95500,3,Vélo,469.084020,13873.881187,3.381058
5918,MOBPRO,RESIDENCE,95500,4,Deux-roues motorisé,154.700896,13873.881187,1.115051


In [17]:
#Age des personnes scolarisées 18-24 ans

ages=table_categories('MOBSCO','AGE_CODE',AGES,'LIEU_ETUDE')
exporter(ages,DOSSIER_PROCESSED/'ages_scolarises_communes_j13d.csv')
age_disponible='MOBSCO' in DETAILS and SOURCES['MOBSCO']['analyser_ages']
indicateurs['J13D_MOBSCO_AGES_DISPONIBLES']=age_disponible
for role,suffixe in [('RESIDENCE','RESIDENTS'),('LIEU_ETUDE','LIEU_ETUDE')]:
    col=f'J13D_MOBSCO_1824_{suffixe}'
    counts=ages[ages['ROLE'].eq(role)&ages['CATEGORIE_CODE'].eq('18')].set_index('CODGEO')['EFFECTIF_PONDERE']
    indicateurs[col]=indicateurs['CODGEO'].map(counts).fillna(0) if age_disponible else np.nan
    decrire(col,'Scolarisés de 18–24 ans — '+role.lower(),'personnes pondérées',
            'Somme IPONDI ; AGEREV10 = 18 ; rôle '+role,'MOBSCO')
display(ages[ages['CODGEO'].isin(FOCUS)])


,THEME,ROLE,CODGEO,CATEGORIE_CODE,CATEGORIE,EFFECTIF_PONDERE,TOTAL_ROLE_COMMUNE,PART_PCT
8859,MOBSCO,RESIDENCE,95127,02,2 ans,227.812151,23560.191641,0.966937
8860,MOBSCO,RESIDENCE,95127,03,3 ans,942.272487,23560.191641,3.999426
8861,MOBSCO,RESIDENCE,95127,04,4 ans,1135.099598,23560.191641,4.817871
8862,MOBSCO,RESIDENCE,95127,05,5 ans,1051.155114,23560.191641,4.461573
8863,MOBSCO,RESIDENCE,95127,06,6–10 ans,5128.562406,23560.191641,21.767915
...,...,...,...,...,...,...,...,...
17321,MOBSCO,LIEU_ETUDE,95500,11,11–14 ans,2849.418097,12129.193798,23.492230
17322,MOBSCO,LIEU_ETUDE,95500,15,15–17 ans,2535.647368,12129.193798,20.905325
17323,MOBSCO,LIEU_ETUDE,95500,18,18–24 ans,2369.357229,12129.193798,19.534334
17324,MOBSCO,LIEU_ETUDE,95500,25,25–29 ans,221.430842,12129.193798,1.825602


In [18]:
#Controler les pondérations


bilan=[]
for theme in DETAILS:
    t=flux[flux['THEME'].eq(theme)];i=indicateurs.set_index('CODGEO')
    c=lambda m:i[f'J13D_{theme}_{m}']
    assert np.allclose(c('RESIDENTS'),c('INTERNES')+c('SORTANTS_FRANCE')+c('SORTANTS_ETRANGER')+c('DEST_NON_LOCALISEES'))
    assert np.allclose(c('LIEU'),c('INTERNES')+c('ENTRANTS'))
    assert np.allclose(c('LIEU'),c('RESIDENTS')-c('SORTANTS_ETRANGER')-c('DEST_NON_LOCALISEES')+c('SOLDE_FRANCE'))
    q=qualite.set_index('THEME').loc[theme]
    assert np.isclose(t['EFFECTIF_PONDERE'].sum(),q['EFFECTIF_PONDERE_RETENU'])
    assert int(t['NB_ENREGISTREMENTS'].sum())==q['NB_LIGNES_RETENUES']
    resident=float(t.loc[t['ORIGINE_IDF'],'EFFECTIF_PONDERE'].sum())
    destination=float(t.loc[t['DESTINATION_IDF'],'EFFECTIF_PONDERE'].sum())
    assert np.isclose(c('RESIDENTS').sum(),resident)
    assert np.isclose(c('LIEU').sum(),destination)
    bilan.append({'THEME':theme,'RESIDENTS_IDF':resident,'LIEUX_IDF':destination,
                  'POIDS_LIENS_CONCERNANT_IDF':float(t['EFFECTIF_PONDERE'].sum()),'ETAT':'COHERENT'})
for t in [modes,ages]:
    if not t.empty:assert np.allclose(t.groupby(['THEME','ROLE','CODGEO'])['PART_PCT'].sum(),100)
for c in indicateurs:
    if c.endswith('_PCT'):assert indicateurs[c].dropna().between(0,100).all()
bilan=pd.DataFrame(bilan)
exporter(bilan,DOSSIER_INTERIM/'bilan_controles_j13d.csv')
display(bilan)


,THEME,RESIDENTS_IDF,LIEUX_IDF,POIDS_LIENS_CONCERNANT_IDF,ETAT
0,MOBPRO,5.677347e+06,5.994480e+06,6.090445e+06,COHERENT
1,MOBSCO,3.231480e+06,3.236787e+06,3.302615e+06,COHERENT


In [19]:
# Comparer Cergy et Pontoise puis leur périmètre commun

profil_focus=indicateurs[indicateurs['CODGEO'].isin(FOCUS)].copy()
flux_focus=flux[flux['ORIGINE'].isin(FOCUS)|flux['DESTINATION'].isin(FOCUS)].copy()
flux_focus['ORIGINE_DANS_FOCUS']=flux_focus['ORIGINE'].isin(FOCUS)
flux_focus['DESTINATION_DANS_FOCUS']=flux_focus['DESTINATION'].isin(FOCUS)
bilan_focus=[]
for theme in SOURCES:
    if theme not in DETAILS:
        bilan_focus.append({'THEME':theme,'SOURCE_CHARGEE':False,'RESIDENTS':np.nan,'INTERNES_FOCUS':np.nan,
                            'ENTRANTS_FOCUS':np.nan,'SORTANTS_FRANCE_FOCUS':np.nan,'SORTANTS_ETRANGER_FOCUS':np.nan,
                            'DEST_NON_LOCALISEES':np.nan,'LIEU_FOCUS':np.nan});continue
    t=flux_focus[flux_focus['THEME'].eq(theme)];o=t['ORIGINE_DANS_FOCUS'];d=t['DESTINATION_DANS_FOCUS']
    connu=t['STATUT_DESTINATION'].eq('COMMUNE_REFERENCEE')
    s=lambda m:float(t.loc[m,'EFFECTIF_PONDERE'].sum())
    r={'THEME':theme,'SOURCE_CHARGEE':True,'RESIDENTS':s(o),'INTERNES_FOCUS':s(o&d),
       'ENTRANTS_FOCUS':s(~o&d),'SORTANTS_FRANCE_FOCUS':s(o&~d&connu),
       'SORTANTS_ETRANGER_FOCUS':s(o&t['STATUT_DESTINATION'].eq('ETRANGER')),
       'DEST_NON_LOCALISEES':s(o&t['STATUT_DESTINATION'].isin(['NON_REFERENCEE','NON_LOCALISEE'])),
       'LIEU_FOCUS':s(d)}
    assert np.isclose(r['LIEU_FOCUS'],r['INTERNES_FOCUS']+r['ENTRANTS_FOCUS'])
    assert np.isclose(r['RESIDENTS'],sum(r[k] for k in ['INTERNES_FOCUS','SORTANTS_FRANCE_FOCUS','SORTANTS_ETRANGER_FOCUS','DEST_NON_LOCALISEES']))
    bilan_focus.append(r)
bilan_focus=pd.DataFrame(bilan_focus)
exporter(bilan_focus,DOSSIER_PROCESSED/'bilan_focus_cergy_pontoise_j13d.csv')
exporter(flux_focus,DOSSIER_PROCESSED/'flux_focus_communes_j13d.csv')
display(profil_focus)
display(bilan_focus)


,CODGEO,NOM_COMMUNE,REG,DEP,J13D_POPULATION_RP_IRIS,J13D_NB_IRIS,J13D_MOBPRO_CHARGE,J13D_MOBPRO_RESIDENTS,J13D_MOBPRO_INTERNES,J13D_MOBPRO_SORTANTS_FRANCE,J13D_MOBPRO_SORTANTS_ETRANGER,J13D_MOBPRO_DEST_NON_LOCALISEES,J13D_MOBPRO_ENTRANTS,J13D_MOBPRO_LIEU,J13D_MOBPRO_SOLDE_FRANCE,J13D_MOBPRO_PART_INTERNES_DEST_CONNUE_PCT,J13D_MOBPRO_PART_ENTRANTS_LIEU_PCT,J13D_MOBPRO_PART_DEST_NON_LOCALISEES_PCT,J13D_MOBPRO_RATIO_LIEU_RESIDENTS,J13D_MOBSCO_CHARGE,J13D_MOBSCO_RESIDENTS,J13D_MOBSCO_INTERNES,J13D_MOBSCO_SORTANTS_FRANCE,J13D_MOBSCO_SORTANTS_ETRANGER,J13D_MOBSCO_DEST_NON_LOCALISEES,J13D_MOBSCO_ENTRANTS,J13D_MOBSCO_LIEU,J13D_MOBSCO_SOLDE_FRANCE,J13D_MOBSCO_PART_INTERNES_DEST_CONNUE_PCT,J13D_MOBSCO_PART_ENTRANTS_LIEU_PCT,J13D_MOBSCO_PART_DEST_NON_LOCALISEES_PCT,J13D_MOBSCO_RATIO_LIEU_RESIDENTS,J13D_ANNEE_RP,J13D_GEOGRAPHIE,J13D_MOBSCO_AGES_DISPONIBLES,J13D_MOBSCO_1824_RESIDENTS,J13D_MOBSCO_1824_LIEU_ETUDE
1120,95127,Cergy,11,95,69578.0,27,True,28449.353783,7832.636126,20588.965150,27.752507,0.00000,25678.690190,33511.326317,5089.725041,27.558743,76.626899,0.000000,1.177929,True,23560.191641,18059.587810,5500.603831,0.0,0.0,20613.491969,38673.079779,15112.888138,76.652975,53.301915,0.0,1.641459,2022,2024,True,6574.85791,19018.552934
1221,95500,Pontoise,11,95,31623.0,10,True,13873.881187,3500.438986,10335.539743,27.808919,10.09354,11040.300707,14540.739692,704.760963,25.299540,75.926679,0.072752,1.048066,True,8888.362612,6240.879257,2647.483354,0.0,0.0,5888.314540,12129.193798,3240.831186,70.214049,48.546628,0.0,1.364615,2022,2024,True,1884.55066,2369.357229


,THEME,SOURCE_CHARGEE,RESIDENTS,INTERNES_FOCUS,ENTRANTS_FOCUS,SORTANTS_FRANCE_FOCUS,SORTANTS_ETRANGER_FOCUS,DEST_NON_LOCALISEES,LIEU_FOCUS
0,MOBPRO,True,42323.234970,13329.168194,34722.897815,28928.411811,55.561425,10.09354,48052.066009
1,MOBSCO,True,32448.554252,25638.232888,25164.040688,6810.321364,0.000000,0.00000,50802.273576


In [20]:
#Principales origines et destinations du focus

partenaires=[]
for theme,t in flux_focus.groupby('THEME'):
    for sens,masque,cle,nom in [
        ('ENTRANTS',~t['ORIGINE_DANS_FOCUS']&t['DESTINATION_DANS_FOCUS'],'ORIGINE','NOM_ORIGINE'),
        ('SORTANTS',t['ORIGINE_DANS_FOCUS']&~t['DESTINATION_DANS_FOCUS'],'DESTINATION','NOM_DESTINATION')]:
        a=t[masque].groupby([cle,nom],dropna=False,as_index=False)['EFFECTIF_PONDERE'].sum()
        a=a.rename(columns={cle:'CODE_PARTENAIRE',nom:'NOM_PARTENAIRE'})
        a['THEME']=theme;a['SENS']=sens
        a['TOTAL_SENS']=float(a['EFFECTIF_PONDERE'].sum())
        a['PART_PCT']=ratio(a['EFFECTIF_PONDERE'],a['TOTAL_SENS'])
        partenaires.append(a)
partenaires=pd.concat(partenaires,ignore_index=True) if partenaires else pd.DataFrame(
    columns=['CODE_PARTENAIRE','NOM_PARTENAIRE','EFFECTIF_PONDERE','THEME','SENS','TOTAL_SENS','PART_PCT'])
partenaires=partenaires.sort_values(['THEME','SENS','EFFECTIF_PONDERE'],ascending=[True,True,False])
exporter(partenaires,DOSSIER_PROCESSED/'partenaires_focus_j13d.csv')
display(partenaires.head(20))


,CODE_PARTENAIRE,NOM_PARTENAIRE,EFFECTIF_PONDERE,THEME,SENS,TOTAL_SENS,PART_PCT
423,75056,Paris,1337.710960,MOBPRO,ENTRANTS,34722.897815,3.852533
898,95572,Saint-Ouen-l'Aumône,1273.556445,MOBPRO,ENTRANTS,34722.897815,3.667771
913,95637,Vauréal,1238.491361,MOBPRO,ENTRANTS,34722.897815,3.566786
881,95476,Osny,1112.230586,MOBPRO,ENTRANTS,34722.897815,3.203162
848,95323,Jouy-le-Moutier,1017.307180,MOBPRO,ENTRANTS,34722.897815,2.929788
824,95218,Éragny,814.363390,MOBPRO,ENTRANTS,34722.897815,2.345321
519,78172,Conflans-Sainte-Honorine,770.475496,MOBPRO,ENTRANTS,34722.897815,2.218926
845,95306,Herblay-sur-Seine,636.482878,MOBPRO,ENTRANTS,34722.897815,1.833035
829,95252,Franconville,608.617174,MOBPRO,ENTRANTS,34722.897815,1.752783
861,95388,Menucourt,578.444258,MOBPRO,ENTRANTS,34722.897815,1.665887


In [21]:
#Joindre le contexte communal à l'IRIS

profil_iris=base.copy()
profil_iris['J13D_COMMUNE_CONTEXTE']=cle_commune
profil_iris['J13D_MAILLE_CONTEXTE']='COMMUNE_SOURCE_'+str(GEOGRAPHIE)
profil_iris['J13D_ANNEE_MOBILITES']=ANNEE_RP
choix=[f'J13D_{theme}_{m}' for theme in SOURCES for m in ['RESIDENTS','LIEU','ENTRANTS','SORTANTS_FRANCE']]
renommage={c:'J13D_CONTEXTE_COMMUNAL_'+c.removeprefix('J13D_') for c in choix}
contexte=indicateurs[['CODGEO']+choix].rename(columns={'CODGEO':'J13D_COMMUNE_CONTEXTE',**renommage})
profil_iris=profil_iris.merge(contexte,on='J13D_COMMUNE_CONTEXTE',how='left',validate='many_to_one',sort=False)
assert len(profil_iris)==NOMBRE_IRIS and set(profil_iris['IRIS'])==ATTENDUS
pd.testing.assert_frame_equal(profil_iris.set_index('IRIS')[base.columns.drop('IRIS')],
                              base.set_index('IRIS')[base.columns.drop('IRIS')],check_dtype=False)
for ancien,nouveau in renommage.items():
    decrire(nouveau,'Contexte communal — '+ancien.removeprefix('J13D_'),'personnes pondérées',
            'Jointure par commune source ; valeur répétée sur chaque IRIS',agregation='NE_PAS_SOMMER_ENTRE_IRIS')
profil_iris_focus=profil_iris[profil_iris['J13D_COMMUNE_CONTEXTE'].isin(FOCUS)].copy()
print('IRIS avant/après :',len(base),len(profil_iris))


IRIS avant/après : 5264 5264


In [23]:
#Exporter les CSV et géographiques

FICHIERS_SORTIE=[]
tables={
 'flux_od_communes_idf_j13d.csv':flux,
 'flux_detail_agrege_idf_j13d.csv':detail,
 'profil_communes_mobilites_idf_j13d.csv':indicateurs,
 'profil_communes_focus_j13d.csv':profil_focus,
 'profil_iris_idf_j13d.csv':profil_iris,
 'profil_iris_focus_j13d.csv':profil_iris_focus,
}
for nom,table in tables.items():FICHIERS_SORTIE.append(exporter(table,DOSSIER_PROCESSED/nom))
geo_mobilites=geo_communes.merge(indicateurs,on='CODGEO',validate='one_to_one')
geo_iris=iris.merge(profil_iris,on='IRIS',validate='one_to_one')
for geo,nom,couche in [(geo_mobilites,'profil_communes_mobilites_idf_j13d.gpkg','communes_mobilites'),
                        (geo_iris,'profil_iris_idf_j13d.gpkg','profils_iris')]:
    final=DOSSIER_PROCESSED/nom;temp=final.with_name(final.stem+'_temp.gpkg')
    if temp.exists():temp.unlink()
    geo.to_file(temp,layer=couche,driver='GPKG',index=False)
    temp.replace(final);FICHIERS_SORTIE.append(final)
print('Exports principaux :',len(FICHIERS_SORTIE))


Exports principaux : 8


In [24]:
#Carte régionaux des lieux de travail et d'études

FICHIERS_HTML=[]
def sauver_figure(fig,nom):
    p=DOSSIER_CARTES/nom
    fig.write_html(p,include_plotlyjs=True,full_html=True,config=CONFIG_PLOTLY)
    FICHIERS_HTML.append(p)
    if AFFICHER_CARTES:fig.show(config=CONFIG_PLOTLY)
    return p
cartes_geo=geo_mobilites.copy()
cartes_geo.geometry=cartes_geo.geometry.simplify(SIMPLIFICATION_METRES,preserve_topology=True)
cartes_geo=cartes_geo.to_crs(4326)
geojson=json.loads(cartes_geo[['CODGEO','geometry']].to_json())
for theme in DETAILS:
    col=f'J13D_{theme}_LIEU'
    titre=('Actifs déclarant un lieu de travail' if theme=='MOBPRO' else 'Personnes scolarisées selon le lieu d’études')
    fig=go.Figure(go.Choropleth(geojson=geojson,featureidkey='properties.CODGEO',
        locations=cartes_geo['CODGEO'],z=cartes_geo[col],text=cartes_geo['NOM_COMMUNE'].map(html.escape),
        colorscale='YlOrRd',colorbar_title='Effectif pondéré',marker_line_width=0.4,
        hovertemplate='%{text}<br>Effectif pondéré : %{z:,.1f}<extra></extra>'))
    fig.update_geos(fitbounds='locations',visible=False,projection_type='mercator')
    fig.update_layout(title=titre+' — RP '+str(ANNEE_RP)+', géographie '+str(GEOGRAPHIE),margin=dict(l=0,r=0,t=60,b=10),height=650)
    sauver_figure(fig,f'01_{theme.lower()}_lieux_communes_idf.html')


In [25]:
#Carte des principaux liens du focus en IDF

centres=geo_communes.set_index('CODGEO').geometry.representative_point()
centres=gpd.GeoSeries(centres,crs=CRS_TRAVAIL).to_crs(4326)
couverture_cartes=[]
for theme in DETAILS:
    t=flux_focus[flux_focus['THEME'].eq(theme)]
    inter=t[t['ORIGINE'].ne(t['DESTINATION'])]
    possible=inter[inter['ORIGINE'].isin(centres.index)&inter['DESTINATION'].isin(centres.index)]
    selection=possible[possible['EFFECTIF_PONDERE'].gt(0)].nlargest(TOP_LIENS_CARTE,'EFFECTIF_PONDERE')
    total=float(inter['EFFECTIF_PONDERE'].sum());trace=float(selection['EFFECTIF_PONDERE'].sum())
    couverture_cartes.append({'THEME':theme,'POIDS_LIENS_INTERCOMMUNAUX_FOCUS':total,
                              'POIDS_TRACE':trace,'NB_LIENS_TRACES':len(selection),
                              'PART_POIDS_TRACE_PCT':100*trace/total if total>0 else np.nan})
    fig=go.Figure(go.Choropleth(geojson=geojson,featureidkey='properties.CODGEO',
        locations=cartes_geo['CODGEO'],z=[0]*len(cartes_geo),showscale=False,
        colorscale=[[0,'#f4f5f7'],[1,'#f4f5f7']],marker_line_color='#adb5bd',marker_line_width=0.3,
        hoverinfo='skip'))
    maximum=float(selection['EFFECTIF_PONDERE'].max()) if not selection.empty else 1
    for r in selection.itertuples():
        a=centres.loc[r.ORIGINE];b=centres.loc[r.DESTINATION]
        texte=html.escape(str(r.NOM_ORIGINE))+' → '+html.escape(str(r.NOM_DESTINATION))
        texte+='<br>Effectif pondéré : '+f'{r.EFFECTIF_PONDERE:,.1f}'
        fig.add_trace(go.Scattergeo(lon=[a.x,b.x],lat=[a.y,b.y],mode='lines',
            line=dict(width=1+5*math.sqrt(r.EFFECTIF_PONDERE/maximum),color='#3157a4'),
            text=[texte,texte],hovertemplate='%{text}<extra></extra>',showlegend=False))
    centres_focus=centres.loc[sorted(FOCUS)]
    fig.add_trace(go.Scattergeo(lon=centres_focus.x,lat=centres_focus.y,mode='markers',
        marker=dict(size=10,color='#d94841'),text=[html.escape(str(ref.loc[c,'NOM_COMMUNE'])) for c in centres_focus.index],
        hovertemplate='%{text}<extra></extra>',name='Communes du focus'))
    fig.update_geos(fitbounds='locations',visible=False,projection_type='mercator')
    fig.update_layout(title=f'{theme} — principaux liens IDF du focus ; tracés schématiques',height=650,
                      margin=dict(l=0,r=0,t=65,b=10))
    sauver_figure(fig,f'02_{theme.lower()}_liens_focus_idf.html')
exporter(pd.DataFrame(couverture_cartes),DOSSIER_INTERIM/'couverture_cartes_flux_j13d.csv')


WindowsPath('C:/Users/almou/OneDrive/GeoMarketing_IDF/data/interim/j13d/couverture_cartes_flux_j13d.csv')

In [26]:
#Graphique des partenaires et modes de transport

for (theme,sens),t in partenaires.groupby(['THEME','SENS']):
    t=t.sort_values('EFFECTIF_PONDERE',ascending=False)
    top=t.head(TOP_PARTENAIRES).copy();reste=float(t.iloc[TOP_PARTENAIRES:]['EFFECTIF_PONDERE'].sum())
    labels=top['NOM_PARTENAIRE'].astype(str).tolist();valeurs=top['EFFECTIF_PONDERE'].tolist()
    if reste>0:labels.append('Autres partenaires');valeurs.append(reste)
    assert np.isclose(sum(valeurs),t['EFFECTIF_PONDERE'].sum())
    fig=go.Figure(go.Bar(x=valeurs,y=[html.escape(x) for x in labels],orientation='h',
                         hovertemplate='%{y}<br>Effectif pondéré : %{x:,.1f}<extra></extra>'))
    fig.update_layout(title=f'{theme} — {sens.lower()} depuis/vers l’extérieur du focus',
                      yaxis=dict(autorange='reversed'),xaxis_title='Personnes pondérées',height=600)
    sauver_figure(fig,f'03_{theme.lower()}_{sens.lower()}_partenaires_focus.html')
if not modes.empty:
    t=modes[modes['CODGEO'].isin(FOCUS)&modes['ROLE'].eq('RESIDENCE')].copy()
    fig=go.Figure()
    for code in sorted(FOCUS):
        a=t[t['CODGEO'].eq(code)]
        fig.add_trace(go.Bar(x=a['CATEGORIE'],y=a['PART_PCT'],name=str(ref.loc[code,'NOM_COMMUNE']),
                            hovertemplate='%{x}<br>Part : %{y:.1f} %<extra>%{fullData.name}</extra>'))
    fig.update_layout(title='Mode principal domicile-travail des résidents du focus',
                      barmode='group',yaxis_title='Part dans toutes les modalités (%)',height=550)
    sauver_figure(fig,'04_modes_residents_focus.html')
print('Cartes/graphiques HTML :',len(FICHIERS_HTML))


Cartes/graphiques HTML : 9


Sujet proposé : Où développer une offre de restauration du midi à Cergy et Pontoise ?
J13d apporte le contexte communal ; J13c localise les restaurants et les points
d’arrêt. Les campus sont localisés à partir d’une source dédiée, pas depuis DCETUF.

| Résultat | Lecture possible | À compléter |
|---|---|---|
| Beaucoup d’actifs déclarent travailler dans la commune | Examiner une offre orientée vers les actifs | Horaires, télétravail, lieux de travail précis, prix et fréquentation à midi |
| Part élevée d’entrants | La commune accueille des actifs résidant ailleurs | Cheminements, dessertes et lieux réellement fréquentés |
| Scolarisés de 18–24 ans dans la commune | Étudier les besoins des jeunes scolarisés | Niveau d’enseignement, campus, effectifs sur site, budget, CROUS |
| Beaucoup de sortants parmi les résidents | Une partie des actifs exerce son emploi ailleurs | Demande du soir, jours non travaillés, habitudes et enquêtes |
| Beaucoup de restaurants autour d’un campus | Décrire l’offre concurrente localisée | Restauration universitaire, prix, horaires, qualité et commerces voisins |

In [27]:
#Dictionnaire, métadonnées

dictionnaire=pd.DataFrame(DICTIONNAIRE)
assert not dictionnaire['COLONNE'].duplicated().any()
FICHIER_DICTIONNAIRE=exporter(dictionnaire,DOSSIER_PROCESSED/'dictionnaire_mobilites_j13d.csv')
exporter(dictionnaire,DOSSIER_DOCS/'dictionnaire_j13d.csv')
fichiers_entree=[FICHIER_PROFIL,FICHIER_GEO,FICHIER_META_J13C,FICHIER_COG]+[Path(c['fichier']) for c in SOURCES.values() if c['actif']]
metadata={'etape':'J13D_MOBILITES_COMMUNES_IDF','execution_utc':datetime.now(timezone.utc).isoformat(),
          'annee_rp':ANNEE_RP,'geographie':GEOGRAPHIE,'maille_flux':'COMMUNE','colonne_effectif_par_source':{t:c['poids'] for t,c in SOURCES.items() if c['actif']},
          'filtre':'origine IDF OU destination IDF','codes_focus':sorted(FOCUS),'nb_iris':NOMBRE_IRIS,
          'sources':[{'theme':theme,'fichier':str(c['fichier']),'membre':LECTURES[theme]['membre'],
                      'annee':c['annee'],'geographie':c['geographie'],'champ':c['champ'],'url':c['url'],
                      'type_fichier':c['type_fichier'],'colonne_effectif':c['poids'],'date_telechargement':c['date_telechargement']}
                     for theme,c in SOURCES.items() if c['actif']],
          'empreintes':[{'fichier':str(p),'sha256':empreinte(p)} for p in dict.fromkeys(fichiers_entree)],
          'qualite':json.loads(qualite.to_json(orient='records')),'cartes':[str(p) for p in FICHIERS_HTML],
          'limites':['Effectifs pondérés de personnes ; pas trajets ou passages par jour.',
                     'Les champs travail/études se recouvrent ; pas de somme de clients uniques.',
                     'Contexte communal répété sur les IRIS, sans répartition des flux.',
                     'Millésime géographique contrôlé ; aucun changement de commune implicitement rétropolé.',
                     'Étranger et destinations non référencées conservés séparément.',
                     'Cartes de liens limitées aux partenaires IDF ; CSV et graphiques gardent les autres.',
                     'Solde France exclut les destinations étrangères et non localisées.',
                     '18–24 ans scolarisés ne signifie pas étudiants du supérieur.',
                     'Pas de présence réelle à midi, de fréquentation ou de prévision de ventes.'],
          'logiciels':{'pandas':pd.__version__,'geopandas':gpd.__version__,'plotly':plotly.__version__}}
FICHIER_METADATA=DOSSIER_PROCESSED/'metadata_mobilites_j13d.json'
FICHIER_METADATA.write_text(json.dumps(metadata,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
doc=f'''# J13d — Mobilités professionnelles et scolaires en Île-de-France

## Objectif
Comparer les communes selon les liens résidence–travail et résidence–études,
et enrichir les profils IRIS par un contexte communal clairement identifié.

## Sources et périmètre
RP {ANNEE_RP}, géographie {GEOGRAPHIE}. Sources MOBPRO/MOBSCO :
{', '.join(t+'='+c['type_fichier']+' ('+c['poids']+')' for t,c in SOURCES.items() if c['actif'])}.
COG {ANNEE_COG} pour les codes et libellés. Focus : {', '.join(sorted(FOCUS))}.
https://www.insee.fr/fr/statistiques/8589904?sommaire=8590191
https://www.insee.fr/fr/statistiques/8589945?sommaire=8590191
https://www.insee.fr/fr/information/7766585

## Méthode
Lecture CSV/ZIP par blocs ; sélection origine OU destination IDF ; somme des poids
sans déduplication des individus en DETAIL, somme des effectifs publiés en FLUX.
Arrondissements ramenés à la commune parente.
Échanges hors IDF, destinations étrangères et destinations non localisées conservés.
Le focus est recalculé sur les paires OD pour éviter les doubles comptages internes.

## Sorties
Matrices OD, indicateurs communaux, modes, âges, focus et profils IRIS de contexte.
CSV et GeoPackages dans data/processed/j13d ; contrôles dans data/interim/j13d.
Cartes Plotly autonomes dans outputs/cartes/j13d ; aucune clé API.

## Validation et limites
Conservation des poids et des enregistrements, identités résidents/lieux, sources
non chargées en NA, préservation des colonnes J13c et couverture géographique.
Les flux ne mesurent pas la fréquentation quotidienne. Les valeurs communales
répétées sur les IRIS ne doivent pas être sommées. Les deux champs ne donnent pas
une clientèle unique. Les cartes de liens sont schématiques, avec couverture exportée.
'''
(DOSSIER_DOCS/'j13d.md').write_text(doc,encoding='utf-8')
print('Métadonnées et documentation créées.')

colonnes_sources=['fichier','source','annee_donnees','annee_geographie','date_telechargement','url']
lignes_sources=[]
for theme,cfg in SOURCES.items():
    if not cfg['actif']:continue
    lignes_sources.append({'fichier':Path(cfg['fichier']).name,
        'source':'Insee - RP '+str(cfg['annee'])+' - '+theme+' - '+cfg['type_fichier'],
        'annee_donnees':cfg['annee'],'annee_geographie':cfg['geographie'],
        'date_telechargement':cfg['date_telechargement'] or '', 'url':cfg['url']})
lignes_sources.append({'fichier':Path(FICHIER_COG).name,'source':'Insee - Code officiel géographique - Communes',
    'annee_donnees':ANNEE_COG,'annee_geographie':ANNEE_COG,
    'date_telechargement':DATE_TELECHARGEMENT_COG or '', 'url':URL_COG})
FICHIER_SOURCES_A_AJOUTER=DOSSIER_DOCS/'sources_a_ajouter_j13d.csv'
pd.DataFrame(lignes_sources,columns=colonnes_sources).to_csv(FICHIER_SOURCES_A_AJOUTER,
    index=False,sep=';',encoding='utf-8-sig')
print('Lignes pour docs/sources.csv :',FICHIER_SOURCES_A_AJOUTER)
display(pd.DataFrame(lignes_sources,columns=colonnes_sources))


Métadonnées et documentation créées.
Lignes pour docs/sources.csv : C:\Users\almou\OneDrive\GeoMarketing_IDF\docs\sources_a_ajouter_j13d.csv


,fichier,source,annee_donnees,annee_geographie,date_telechargement,url
0,FD_MOBPRO_2022.csv,Insee - RP 2022 - MOBPRO - DETAIL,2022,2024,,https://www.insee.fr/fr/statistiques/8589904?s...
1,FD_MOBSCO_2022.csv,Insee - RP 2022 - MOBSCO - DETAIL,2022,2024,,https://www.insee.fr/fr/statistiques/8589945?s...
2,v_commune_2024.csv,Insee - Code officiel géographique - Communes,2024,2024,,https://www.insee.fr/fr/information/7766585


In [28]:
#Controles finaux

assert set(indicateurs['CODGEO'])==CODES_IDF and indicateurs['CODGEO'].is_unique
assert len(profil_iris)==NOMBRE_IRIS and set(profil_iris['IRIS'])==ATTENDUS
assert set(profil_iris['J13D_COMMUNE_CONTEXTE'])==CODES_IDF
assert not pd.DataFrame(DICTIONNAIRE)['COLONNE'].duplicated().any()
for theme,cfg in SOURCES.items():
    colonnes=[f'J13D_{theme}_{m}' for m in MESURES]
    if not cfg['actif']:assert indicateurs[colonnes].isna().all().all()
    else:
        for m in ['RESIDENTS','INTERNES','SORTANTS_FRANCE','SORTANTS_ETRANGER','DEST_NON_LOCALISEES','ENTRANTS','LIEU']:
            assert indicateurs[f'J13D_{theme}_{m}'].ge(0).all()
        vide=indicateurs[f'J13D_{theme}_RESIDENTS'].eq(0)
        assert indicateurs.loc[vide,f'J13D_{theme}_RATIO_LIEU_RESIDENTS'].isna().all()
assert all(p.is_file() for p in FICHIERS_SORTIE+[FICHIER_DICTIONNAIRE,FICHIER_METADATA,FICHIER_SOURCES_A_AJOUTER])
assert all(p.is_file() for p in FICHIERS_HTML)
print('J13d : contrôles validés ;',len(indicateurs),'communes et',len(profil_iris),'IRIS conservés.')
print('Focus :',', '.join(str(ref.loc[c,'NOM_COMMUNE']) for c in sorted(FOCUS)))


J13d : contrôles validés ; 1267 communes et 5264 IRIS conservés.
Focus : Cergy, Pontoise
